In [1]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

# Global seaborn / matplotlib defaults
sns.set_theme(
    style="whitegrid",          # axes with grid
    # palette="colorblind",            # default colour palette
    # font_scale=1.3,             # scales all font sizes uniformly
    rc={
        # "lines.linewidth": 4,           # default line width
        # "axes.titlesize": 16,
        # "axes.labelsize": 22,
        # "axes.labelweight": "bold",     
        # "xtick.labelsize": 18,
        # "ytick.labelsize": 18,
        # "legend.fontsize": 19,
        "grid.linestyle": "-",
        "grid.alpha": 0.6,
        # "lines.markersize": 8,

    },
)

In [3]:
import pandas as pd

from analysis.utils.stats import (
    KNOWN_METRIC_KEYWORDS,
    compute_clean_dirty_difference_statistics,
    compute_kl_target_correlations,
    compute_metric_statistics_by_group,
    get_benchmark_columns_by_keywords,
    summarize_kl_target_correlations,
)
from analysis.utils.vis import (
    plot_difference_statistics,
    plot_grouped_difference_statistics,
    plot_kl_correlation_barplot,
    plot_metrics_correlation,
    plot_run_comparisons,
)


def preview_results(df: pd.DataFrame, sample_size: int = 10) -> None:
    if len(df) > 0:
        display(df.sample(min(sample_size, len(df))))

clean_results_path = "../clean_results/greedy/results.json"
clean_results = pd.read_json(clean_results_path, orient="records")

preview_results(clean_results)

,path,file,benchmark,logprobs,sample_index,gold_index,choices,choice_lengths,doc_index
282474,/home/fre.gilad/source/silent_direction/clean_...,mastermind_easy.json,mastermind_24_easy,"[-36.25, -29.75, -28.75, -38.0]",1024,2,"[brown, black, red, green, red, black, black, ...","[12.0, 10.0, 10.0, 12.0]",1024
234972,/home/fre.gilad/source/silent_direction/clean_...,mastermind_easy.json,mastermind_35_easy,"[-56.5, -44.75, -60.25, -63.75]",950,1,"[red, green, pink, pink, black, red, pink, gre...","[16.0, 16.0, 16.0, 18.0]",950
200337,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_anaphor_number_agreement,"[-100.0, -118.5]",337,0,"[These guests haven't approached themselves., ...","[43.0, 40.0]",337
150415,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_adjunct_island,"[-85.5, -90.0]",365,0,[What had Roger ascended while noticing steps?...,"[45.0, 45.0]",365
99805,/home/fre.gilad/source/silent_direction/clean_...,anli.json,anli_r2,"[-4.09375, -0.59375, -0.8437500000000001]",905,1,"[True, Neither, False]","[4.0, 7.0, 5.0]",905
269902,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_principle_A_case_1,"[-89.5, -104.5]",452,0,"[Samuel thinks that he loved Christina., Samue...","[38.0, 43.0]",452
87516,/home/fre.gilad/source/silent_direction/clean_...,mastermind_easy.json,mastermind_35_easy,"[-37.0, -38.5, -30.5, -29.75]",344,2,"[white, white, orange, purple, brown, purple, ...","[20.0, 21.0, 20.0, 19.0]",344
64679,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_existential_there_subject_raising,"[-118.5, -129.0]",29,0,[There aren't sure to be few bicycles boring H...,"[50.0, 55.0]",29
13209,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_ellipsis_n_bar_1,"[-163.0, -171.0]",9,0,[This print scares a lot of busy senators and ...,"[67.0, 67.0]",9
331942,/home/fre.gilad/source/silent_direction/clean_...,mastermind_easy.json,mastermind_35_easy,"[-42.5, -26.5, -34.5, -29.625]",20,1,"[brown, brown, yellow, black, white, pink, bro...","[20.0, 18.0, 18.0, 19.0]",20


In [4]:
RELEVANT_FILES = [
    "anli",
    "blimp",
    "mastermind_easy",
    "metabench_arc",
    "metabench_hellaswag",
    "metabench_mmlu",
    "metabench_truthfulqa",
    "metabench_winogrande",
    "piqa",
    "toxigen",
    "wmdp",
]

In [5]:
def filter_files(df: pd.DataFrame, relevant_files: list[str]) -> pd.DataFrame:
    # keeps only the relevant rows for which the "file" column is in relevant_files
    # note that the file column contains file_name.json, so we check if any of the relevant_files is a substring
    
    out = df.copy()
    out = out[out["file"].apply(lambda x: any(rel_file in x for rel_file in relevant_files))]
    return out


clean_results = filter_files(clean_results, RELEVANT_FILES)

In [6]:
BENCHMARK_METRIC_SEP = " / "

def add_clean_model_name(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    parts = out["path"].astype(str).str.split("/")
    if (parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")
    out["model_name"] = parts.str[-2]
    return out

clean_results = add_clean_model_name(clean_results)
clean_results.head()

,path,file,benchmark,logprobs,sample_index,gold_index,choices,choice_lengths,doc_index,model_name
0,/home/fre.gilad/source/silent_direction/clean_...,anli.json,anli_r1,"[-21.375, -30.75, -16.5]",0,0,"[True, Neither, False]","[4.0, 7.0, 5.0]",0,gemma-2b-it
1,/home/fre.gilad/source/silent_direction/clean_...,anli.json,anli_r1,"[-17.375, -32.0, -18.125]",1,0,"[True, Neither, False]","[4.0, 7.0, 5.0]",1,gemma-2b-it
2,/home/fre.gilad/source/silent_direction/clean_...,anli.json,anli_r1,"[-21.0, -35.75, -19.875]",2,0,"[True, Neither, False]","[4.0, 7.0, 5.0]",2,gemma-2b-it
3,/home/fre.gilad/source/silent_direction/clean_...,anli.json,anli_r1,"[-14.75, -38.5, -21.75]",3,1,"[True, Neither, False]","[4.0, 7.0, 5.0]",3,gemma-2b-it
4,/home/fre.gilad/source/silent_direction/clean_...,anli.json,anli_r1,"[-15.125, -37.5, -19.625]",4,2,"[True, Neither, False]","[4.0, 7.0, 5.0]",4,gemma-2b-it


In [7]:
clean_results.sample(10)

,path,file,benchmark,logprobs,sample_index,gold_index,choices,choice_lengths,doc_index,model_name
267548,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_only_npi_scope,"[-178.0, -186.0]",98,0,[Only waiters that Rose discovered have ever r...,"[89.0, 89.0]",98,Qwen2.5-3B-Instruct
158881,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_distractor_agreement_relational_noun,"[-84.5, -85.5]",331,0,[The grandmothers of Gary aren't managing to s...,"[65.0, 64.0]",331,Qwen2.5-0.5B-Instruct
77118,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_regular_plural_subject_verb_agreement_1,"[-86.0, -87.5]",468,0,"[The Lutherans reference Jason., The Lutherans...","[30.0, 31.0]",468,Llama-2-7b-chat-hf
153801,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_coordinate_structure_constraint_complex_...,"[-88.5, -97.0]",251,0,[Whose octopi had Wayne discovered and Mitchel...,"[54.0, 54.0]",251,Qwen2.5-0.5B-Instruct
238132,/home/fre.gilad/source/silent_direction/clean_...,metabench_truthfulqa.json,metabench_truthfulqa,"[-43.25, -72.5, -85.0, -109.5]",115,0,"[No, lawyers are not allowed to lie., Yes, law...","[35.0, 48.0, 57.0, 65.0]",115,Qwen2.5-14B-Instruct
266837,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_npi_present_2,"[-121.5, -125.0]",387,0,[Those organizations clearly motivate Teresa's...,"[65.0, 62.0]",387,Qwen2.5-3B-Instruct
299942,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_complex_NP_island,"[-144.0, -140.0]",42,0,[Who are Gina's best friends that are boasting...,"[74.0, 74.0]",42,gemma-2b-it
240635,/home/fre.gilad/source/silent_direction/clean_...,toxigen.json,toxigen,"[-31.25, -18.75]",493,1,"[No, Yes]","[2.0, 3.0]",493,Qwen2.5-14B-Instruct
3077,/home/fre.gilad/source/silent_direction/clean_...,anli.json,anli_r3,"[-16.875, -27.75, -16.125]",1077,1,"[True, Neither, False]","[4.0, 7.0, 5.0]",1077,gemma-2b-it
33814,/home/fre.gilad/source/silent_direction/clean_...,blimp.json,blimp_wh_questions_subject_gap,"[-125.0, -144.0]",114,0,[Those pedestrians see the mushroom that distr...,"[56.0, 55.0]",114,gemma-2b-it


In [8]:
print(clean_results.columns.tolist())

['path', 'file', 'benchmark', 'logprobs', 'sample_index', 'gold_index', 'choices', 'choice_lengths', 'doc_index', 'model_name']


In [9]:
# compute logprobs_normalized
# each entry in logprobs column contains list[float]
# each entry in num_tokens contains list[int]
# logprobs_normalized = logprob[i] / num_tokens[i]

clean_results["logprobs_normalized"] = clean_results.apply(
    lambda row: [lp / nt if nt > 0 else float('nan') for lp, nt in zip(row["logprobs"], row["choice_lengths"])], axis=1
)

In [10]:
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import poisson_binom


def build_gt_distributions(
    df: pd.DataFrame,
    *,
    logprobs_col: str = "logprobs",
    benchmark_col: str = "benchmark",
    model_col: str = "model_name",
    gold_index_col: str = "gold_index",
    sample_index_col: str = "sample_index",
):
    """
    Returns a dict:
        (benchmark, model_name) -> {
            "n": int,
            "mean": float,
            "var": float,
            "std": float,
            "dist": poisson_binom object
        }
    """

    def gold_prob(logprobs, gold_index):
        arr = np.asarray(logprobs, dtype=np.float64)
        return float(np.exp(arr[gold_index] - logsumexp(arr)))

    df = df.copy()
    df["_p"] = [
        gold_prob(lp, gi)
        for lp, gi in zip(df[logprobs_col], df[gold_index_col])
    ]

    out = {}

    for (benchmark, model_name), g in df.groupby([benchmark_col, model_col], sort=False):
        g = g.sort_values(sample_index_col)

        p = g["_p"].to_numpy()
        n = len(p)

        mean = float(np.mean(p))
        var = float(np.sum(p * (1 - p)) / (n * n))
        std = float(np.sqrt(var))

        out[(benchmark, model_name)] = {
            "n": n,
            "mean": mean,
            "var": var,
            "std": std,
            "dist": poisson_binom(p),
        }

    return out

def upper_tail(d: dict, s: float):
    n = d["n"]
    k = int(np.ceil(s * n))
    if k <= 0:
        return 1.0
    if k > n:
        return 0.0
    return float(d["dist"].sf(k - 1))  # P(X >= k)


def lower_tail(d: dict, s: float):
    n = d["n"]
    k = int(np.floor(s * n))
    if k < 0:
        return 0.0
    if k >= n:
        return 1.0
    return float(d["dist"].cdf(k))  # P(X <= k)


def two_sided_tail(d: dict, s: float):
    mean = d["mean"]
    delta = abs(s - mean)
    return min(
        1.0,
        lower_tail(d, mean - delta) + upper_tail(d, mean + delta),
    )
    
    

In [11]:
clean_results.head()

,path,file,benchmark,logprobs,sample_index,gold_index,choices,choice_lengths,doc_index,model_name,logprobs_normalized
0,/home/fre.gilad/source/silent_direction/clean_...,anli.json,anli_r1,"[-21.375, -30.75, -16.5]",0,0,"[True, Neither, False]","[4.0, 7.0, 5.0]",0,gemma-2b-it,"[-5.34375, -4.392857142857143, -3.3]"
1,/home/fre.gilad/source/silent_direction/clean_...,anli.json,anli_r1,"[-17.375, -32.0, -18.125]",1,0,"[True, Neither, False]","[4.0, 7.0, 5.0]",1,gemma-2b-it,"[-4.34375, -4.571428571428571, -3.625]"
2,/home/fre.gilad/source/silent_direction/clean_...,anli.json,anli_r1,"[-21.0, -35.75, -19.875]",2,0,"[True, Neither, False]","[4.0, 7.0, 5.0]",2,gemma-2b-it,"[-5.25, -5.107142857142857, -3.975]"
3,/home/fre.gilad/source/silent_direction/clean_...,anli.json,anli_r1,"[-14.75, -38.5, -21.75]",3,1,"[True, Neither, False]","[4.0, 7.0, 5.0]",3,gemma-2b-it,"[-3.6875, -5.5, -4.35]"
4,/home/fre.gilad/source/silent_direction/clean_...,anli.json,anli_r1,"[-15.125, -37.5, -19.625]",4,2,"[True, Neither, False]","[4.0, 7.0, 5.0]",4,gemma-2b-it,"[-3.78125, -5.357142857142857, -3.925]"


### Load ablated data

In [12]:
import pandas as pd

from analysis.utils.stats import (
    KNOWN_METRIC_KEYWORDS,
    compute_clean_dirty_difference_statistics,
    compute_kl_target_correlations,
    compute_metric_statistics_by_group,
    get_benchmark_columns_by_keywords,
    summarize_kl_target_correlations,
)
from analysis.utils.vis import (
    plot_difference_statistics,
    plot_grouped_difference_statistics,
    plot_kl_correlation_barplot,
    plot_metrics_correlation,
    plot_run_comparisons,
)


# clean_results_path = "../clean_results/greedy/results.json"
dirty_results_path = "../logs/silent-norm-runs-v1/results.json"
# dirty_results_path = "../logs/silent-norm-runs-v2/results.json"
# dirty_results_path = "../logs/silent-norm-runs-v3/results.json"
# dirty_results_path = "../logs/random_results.json"

dirty_results = pd.read_json(dirty_results_path, orient="records")

In [13]:
dirty_results.head(20)

,path,file,benchmark,metric,value
0,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r1,"acc,none",0.353000
1,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r2,"acc,none",0.351000
2,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r3,"acc,none",0.361667
3,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp,"acc,none",0.796955
4,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_adjunct_island,"acc,none",0.886000
5,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_anaphor_gender_agreement,"acc,none",0.990000
6,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_anaphor_number_agreement,"acc,none",0.994000
7,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_animate_subject_passive,"acc,none",0.728000
8,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_animate_subject_trans,"acc,none",0.882000
9,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_causative,"acc,none",0.656000


In [14]:
def add_path_metadata(dirty_df: pd.DataFrame) -> pd.DataFrame:
    dirty_df = dirty_df.copy()
    dirty_parts = dirty_df["path"].str.split("/")
    if (dirty_parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")
    
    dirty_out = dirty_df.copy()

    dirty_out["model_name"] = dirty_parts.str[-5]
    dirty_out["train_dataset"] = dirty_parts.str[-4]
    dirty_out["layer_name"] = dirty_parts.str[-3]
    dirty_out["exp_name"] = dirty_parts.str[-2]

    return dirty_out

# apply formatting to metric column
def format_metric_column(df: pd.DataFrame, metric_col: str = "metric") -> pd.DataFrame:
    df = df.copy()
    df[metric_col] = df[metric_col].apply(lambda x: x.replace(",none", ""))
    return df



In [15]:
dirty_results = add_path_metadata(dirty_results)
dirty_results = format_metric_column(dirty_results)

In [16]:
dirty_results.head(10)

,path,file,benchmark,metric,value,model_name,train_dataset,layer_name,exp_name
0,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r1,acc,0.353000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
1,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r2,acc,0.351000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
2,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r3,acc,0.361667,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
3,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp,acc,0.796955,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
4,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_adjunct_island,acc,0.886000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
5,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_anaphor_gender_agreement,acc,0.990000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
6,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_anaphor_number_agreement,acc,0.994000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
7,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_animate_subject_passive,acc,0.728000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
8,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_animate_subject_trans,acc,0.882000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
9,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_causative,acc,0.656000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1


In [17]:
# Build ground truth distributions for both metrics from clean results
print("Building clean distributions for acc...")
dist_acc = build_gt_distributions(clean_results, logprobs_col="logprobs")

print("Building clean distributions for acc_norm...")
dist_acc_norm = build_gt_distributions(clean_results, logprobs_col="logprobs_normalized")

Building clean distributions for acc...
Building clean distributions for acc_norm...


In [18]:


def calculate_statistical_tails(row):
    benchmark = row.get("benchmark")
    model = row.get("model_name")
    metric = row.get("metric")
    score = row.get("value")  # Assuming the result is in the "value" column
    
    # Select the appropriate distribution dictionary based on the metric
    if metric == "acc":
        dists = dist_acc
    elif metric == "acc_norm":
        dists = dist_acc_norm
    else:
        return pd.Series({"upper_tail": float('nan'), "lower_tail": float('nan'), "two_sided_tail": float('nan')})
    
    dist_key = (benchmark, model)
    if dist_key not in dists or pd.isna(score):
        return pd.Series({"upper_tail": float('nan'), "lower_tail": float('nan'), "two_sided_tail": float('nan')})
        
    d = dists[dist_key]
    
    return pd.Series({
        "upper_tail": upper_tail(d, score),
        "lower_tail": lower_tail(d, score),
        "two_sided_tail": two_sided_tail(d, score)
    })

print("Calculating tail probabilities for dirty_results...")
tail_probs = dirty_results.apply(calculate_statistical_tails, axis=1)
dirty_results = pd.concat([dirty_results, tail_probs], axis=1)

dirty_results.head()

Calculating tail probabilities for dirty_results...


Calculating tail probabilities for dirty_results...


,path,file,benchmark,metric,value,model_name,train_dataset,layer_name,exp_name,upper_tail,lower_tail,two_sided_tail
0,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r1,acc,0.353000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,1.000000,2.467272e-09,4.480651e-09
1,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r2,acc,0.351000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.999999,7.715613e-07,1.197154e-06
2,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r3,acc,0.361667,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.883097,1.169028e-01,2.268025e-01
3,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp,acc,0.796955,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,NaN,NaN,NaN
4,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_adjunct_island,acc,0.886000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.000153,9.999346e-01,4.017148e-04


In [19]:
dirty_results['model_name'].unique()

array(['Llama-2-7b-chat-hf', 'Phi-3-mini-4k-instruct',
       'Qwen2.5-3B-Instruct', 'gemma-2b-it'], dtype=object)

In [20]:
mask = (
    (dirty_results["model_name"] == "Llama-2-7b-chat-hf") &
    (dirty_results["layer_name"] == "model.layers.21") 
    # (dirty_results["kl_value"] == 4.0)
)
dirty_results[mask]['exp_name'].unique()

array(['Llama-2-7b-chat-hf-KL-0.0-iter1',
       'Llama-2-7b-chat-hf-KL-0.125-iter1',
       'Llama-2-7b-chat-hf-KL-0.25-iter1',
       'Llama-2-7b-chat-hf-KL-0.5-iter1',
       'Llama-2-7b-chat-hf-KL-1.0-iter1',
       'Llama-2-7b-chat-hf-KL-2.0-iter1',
       'Llama-2-7b-chat-hf-KL-4.0-iter1',
       'Llama-2-7b-chat-hf-KL-8.0-iter1'], dtype=object)

In [41]:
dist_acc_norm[('piqa', 'Llama-2-7b-chat-hf')]

{'n': 1838,
 'mean': 0.5250992391601159,
 'var': 0.00013217287695184042,
 'std': 0.011496646334990061,
 'dist': <scipy.stats._discrete_distns.poisson_binomial_frozen at 0x17af68d10>}

In [42]:
two_sided_tail(dist_acc_norm[('piqa', 'Llama-2-7b-chat-hf')], 0.732862)

2.2625538015143725e-74

In [36]:
exp_mask = (
    (dirty_results["exp_name"] == "Llama-2-7b-chat-hf-KL-4.0-iter1") &
    (dirty_results["layer_name"] == "model.layers.21")
    )

dirty_results[exp_mask].sort_values(["two_sided_tail"], ascending=True)

,path,file,benchmark,metric,value,model_name,train_dataset,layer_name,exp_name,upper_tail,lower_tail,two_sided_tail
3159,/home/fre.gilad/source/silent_direction/logs/s...,piqa.json,piqa,acc_norm,0.732862,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.layers.21,Llama-2-7b-chat-hf-KL-4.0-iter1,0.000000e+00,1.0,2.262554e-74
3094,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_principle_A_domain_2,acc,0.848000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.layers.21,Llama-2-7b-chat-hf-KL-4.0-iter1,2.280398e-13,1.0,6.289445e-12
3090,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_principle_A_c_command,acc,0.806000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.layers.21,Llama-2-7b-chat-hf-KL-4.0-iter1,1.304561e-08,1.0,5.066881e-08
3102,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_superlative_quantifiers_1,acc,0.840000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.layers.21,Llama-2-7b-chat-hf-KL-4.0-iter1,8.438437e-08,1.0,3.297985e-07
3062,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_determiner_noun_agreement_with_adj_irreg...,acc,0.918000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.layers.21,Llama-2-7b-chat-hf-KL-4.0-iter1,3.986756e-08,1.0,6.329693e-07
...,...,...,...,...,...,...,...,...,...,...,...,...
3174,/home/fre.gilad/source/silent_direction/logs/s...,xquad_es.json,xquad_es,f1,0.157465,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.layers.21,Llama-2-7b-chat-hf-KL-4.0-iter1,NaN,NaN,NaN
3175,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,exact_match,0.015126,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.layers.21,Llama-2-7b-chat-hf-KL-4.0-iter1,NaN,NaN,NaN
3176,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,f1,0.125845,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.layers.21,Llama-2-7b-chat-hf-KL-4.0-iter1,NaN,NaN,NaN
3177,/home/fre.gilad/source/silent_direction/logs/s...,xquad_zh.json,xquad_zh,exact_match,0.000000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.layers.21,Llama-2-7b-chat-hf-KL-4.0-iter1,NaN,NaN,NaN
